# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

I chose **Logistic Regression as the primary model**, followed by a constrained **Decision Tree** and **Random Forest** as stronger comparison models.

This is a binary prediction problem: whether an observed content item is labeled as declining. Logistic Regression is a good first learned model because it is simple, reproducible, and interpretable. Its probabilities can also be used to rank pages, which matches the practical refresh-review use case.

The tree models are included to test whether nonlinear relationships and feature interactions add enough value to justify extra complexity. I will not select a model just because it is more complex; the comparison uses the same held-out data and ranking metrics as the baseline.

**Important leakage decision:** `trend_direction` and `trend_pct` are excluded because they define the target. The `*_last_30d` and `*_prev_30d` columns are also excluded because together they directly form the trend label. `content_id` and `client_id` are used only for identification/grouped splitting, never as features.


In [6]:
!git clone https://github.com/ShahvezAli784/-flyrank-machine-learning

Cloning into '-flyrank-machine-learning'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 137 (delta 49), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 1.86 MiB | 9.80 MiB/s, done.
Resolving deltas: 100% (49/49), done.


In [10]:
from pathlib import Path

REPO_ROOT = Path("/content/-flyrank-machine-learning")
DATA_PATH = REPO_ROOT / "data/raw/content_refresh_anonymized.csv"

print("Repo exists:", REPO_ROOT.exists())
print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH)

Repo exists: True
Dataset exists: True
Dataset path: /content/-flyrank-machine-learning/data/raw/content_refresh_anonymized.csv


In [12]:
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_STATE = 42

# Explicit Colab repository location
REPO_ROOT = Path("/content/-flyrank-machine-learning")

# Dataset path
DATA_PATH = REPO_ROOT / "data/raw/content_refresh_anonymized.csv"

# Output directory
OUTPUT_DIR = REPO_ROOT / "work/outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify paths
print("Repository:", REPO_ROOT)
print("Dataset:", DATA_PATH)
print("Repository exists:", REPO_ROOT.exists())
print("Dataset exists:", DATA_PATH.exists())

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH}\n"
        "Make sure the FlyRank repository exists at /content/-flyrank-machine-learning."
    )

# Load dataset
df = pd.read_csv(DATA_PATH)

# Create target
df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# Match the Week-4 baseline population
model_df = (
    df[
        (df["impressions_90d"] > 0)
        & (df["content_age_days"] >= 90)
    ]
    .drop_duplicates(subset=["content_id"])
    .reset_index(drop=True)
)

print(f"Rows used for ML-08: {len(model_df):,}")
print(f"Clients represented: {model_df['client_id'].nunique():,}")
print(
    f"Declining-label rate: "
    f"{model_df['is_declining_label'].mean():.3%}"
)

Repository: /content/-flyrank-machine-learning
Dataset: /content/-flyrank-machine-learning/data/raw/content_refresh_anonymized.csv
Repository exists: True
Dataset exists: True
Rows used for ML-08: 30,000
Clients represented: 32
Declining-label rate: 54.207%


### Feature contract

The model uses decision-time content, keyword, traffic, engagement, position, and freshness signals. Missing numeric values are median-imputed **with missingness indicators**, rather than blindly converting missing values to zero. Categorical missing values are imputed to the most frequent training category and unseen categories are ignored safely by the encoder.

Excluded from features:
- `content_id`, `client_id` — identifiers only
- `trend_direction`, `trend_pct` — target-derived
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`
- `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` — these windows directly determine the observed trend label
- `provider_used`, `model_used` — not used for this decision-support model


In [13]:
NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "is_declining_label"

assert not set(["content_id", "client_id", "trend_direction", "trend_pct"]).intersection(FEATURES)
assert not set([
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"
]).intersection(FEATURES)

print(f"Numeric features: {len(NUMERIC_FEATURES)}")
print(f"Categorical features: {len(CATEGORICAL_FEATURES)}")
print(f"Total raw model features: {len(FEATURES)}")


Numeric features: 23
Categorical features: 9
Total raw model features: 32


## 2. Split design

A **client-held-out split** is more honest than randomly mixing rows from the same client into train and test. Content items from the same client can share patterns, so holding out entire clients reduces the chance that the model benefits from client-specific repetition.

I use a deterministic 80/20 client split with `random_state=42`. If the grouped split did not contain both target classes in both partitions, the notebook would fail rather than silently switching to a weaker evaluation design.


In [14]:
# Deterministic client-held-out split. Client IDs are used only to form the split.
client_series = model_df["client_id"].fillna("unknown").astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()

rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.20)))
test_clients = set(shuffled_clients[:test_client_count])

test_mask = client_series.isin(test_clients).to_numpy()
train_indices = np.flatnonzero(~test_mask)
test_indices = np.flatnonzero(test_mask)

y = model_df[TARGET].astype(int)

if y.iloc[train_indices].nunique() != 2 or y.iloc[test_indices].nunique() != 2:
    raise ValueError("Client-held-out split must contain both target classes in train and test.")

X = model_df[FEATURES].copy()
y_train = y.iloc[train_indices]
y_test = y.iloc[test_indices]

print(f"Train rows: {len(train_indices):,}")
print(f"Test rows: {len(test_indices):,}")
print(f"Train clients: {len(set(client_series.iloc[train_indices])):,}")
print(f"Test clients: {len(test_clients):,}")
print(f"Train declining rate: {y_train.mean():.3%}")
print(f"Test declining rate: {y_test.mean():.3%}")


Train rows: 27,675
Test rows: 2,325
Train clients: 26
Test clients: 6
Train declining rate: 55.476%
Test declining rate: 39.097%


The same held-out test set is used for comparing the baseline and ML models.

## 3. Train + compare vs my baseline

The Week-4 rule is reproduced inside this notebook and evaluated **only on the ML-08 held-out test rows**. This gives the rule and the learned models the same test population and the same ranking metrics.

**Week-4 rule:**

`score = 2 × stale_visible + 0.5 × visibility_percentile + 0.5 × freshness_risk_percentile`

where stale means `days_since_last_update >= 180` and visible means `impressions_90d >= 500`.

The primary comparison metric is **Precision@50**, because the practical use case is ranking a small refresh-review queue. Precision@10 and Precision@20 are also reported, along with ROC-AUC, PR-AUC, precision, recall, and F1 for the learned classifiers.


In [15]:
def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({"y": np.asarray(y_true), "score": np.asarray(scores)})
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0

# Rebuild the Week-4 baseline on the held-out test population only.
test_frame = model_df.iloc[test_indices].copy()
test_frame["visibility_percentile"] = np.log1p(test_frame["impressions_90d"]).rank(
    method="average", pct=True
)
test_frame["freshness_risk_percentile"] = test_frame["days_since_last_update"].rank(
    method="average", pct=True
)
test_frame["stale_visible"] = (
    (test_frame["days_since_last_update"] >= 180)
    & (test_frame["impressions_90d"] >= 500)
).astype(int)
test_frame["baseline_score"] = (
    2.0 * test_frame["stale_visible"]
    + 0.5 * test_frame["visibility_percentile"]
    + 0.5 * test_frame["freshness_risk_percentile"]
)

baseline_scores = test_frame["baseline_score"].to_numpy()

baseline_metrics = {
    "Precision@10": precision_at_k(y_test, baseline_scores, 10),
    "Precision@20": precision_at_k(y_test, baseline_scores, 20),
    "Precision@50": precision_at_k(y_test, baseline_scores, 50),
}

print("Week-4 baseline on the ML-08 held-out test set:")
for metric, value in baseline_metrics.items():
    print(f"{metric}: {value:.3f}")


Week-4 baseline on the ML-08 held-out test set:
Precision@10: 0.700
Precision@20: 0.350
Precision@50: 0.260


In [16]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median", add_indicator=True),
        ),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE,
    ),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        max_depth=5,
        min_samples_leaf=50,
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced_subsample",
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=25,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}

results = []
trained_models = {}
probabilities = {}

for name, estimator in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )
    pipeline.fit(X.iloc[train_indices], y.iloc[train_indices])
    prob = pipeline.predict_proba(X.iloc[test_indices])[:, 1]
    pred = (prob >= 0.5).astype(int)

    probabilities[name] = prob
    trained_models[name] = pipeline
    results.append({
        "Model": name,
        "Precision@10": precision_at_k(y_test, prob, 10),
        "Precision@20": precision_at_k(y_test, prob, 20),
        "Precision@50": precision_at_k(y_test, prob, 50),
        "ROC-AUC": roc_auc_score(y_test, prob),
        "PR-AUC": average_precision_score(y_test, prob),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
    })

comparison = pd.DataFrame(results)
baseline_row = pd.DataFrame([{
    "Model": "Week-4 Baseline",
    "Precision@10": baseline_metrics["Precision@10"],
    "Precision@20": baseline_metrics["Precision@20"],
    "Precision@50": baseline_metrics["Precision@50"],
    "ROC-AUC": np.nan,
    "PR-AUC": np.nan,
    "Precision": np.nan,
    "Recall": np.nan,
    "F1": np.nan,
}])

comparison_table = pd.concat([baseline_row, comparison], ignore_index=True)
comparison_table = comparison_table.sort_values(
    ["Precision@50", "PR-AUC"], ascending=False, na_position="last"
).reset_index(drop=True)

display(comparison_table.round(3))


,Model,Precision@10,Precision@20,Precision@50,ROC-AUC,PR-AUC,Precision,Recall,F1
0,Logistic Regression,0.8,0.80,0.82,0.731,0.626,0.647,0.570,0.606
1,Random Forest,0.8,0.70,0.72,0.746,0.607,0.566,0.741,0.642
2,Decision Tree,0.6,0.45,0.58,0.742,0.575,0.569,0.716,0.634
3,Week-4 Baseline,0.7,0.35,0.26,NaN,NaN,NaN,NaN,NaN


### Model selection

I select the model by **Precision@50**, because the intended use is a ranked review queue. If two models are close, PR-AUC and simplicity are secondary considerations. A model that is more complex but does not improve the decision-relevant ranking metric does not earn complexity.


In [17]:
learned_comparison = comparison[comparison["Model"] != "Week-4 Baseline"].copy()
best_row = learned_comparison.sort_values(
    ["Precision@50", "PR-AUC", "ROC-AUC"], ascending=False
).iloc[0]
best_model_name = best_row["Model"]

print(f"Selected learned model: {best_model_name}")
print(f"Selection metric — Precision@50: {best_row['Precision@50']:.3f}")
print("The selection is based on held-out ranking performance, not model complexity.")


Selected learned model: Logistic Regression
Selection metric — Precision@50: 0.820
The selection is based on held-out ranking performance, not model complexity.


## 4. Errors and interpretation

A score alone is not enough. I inspect false positives, false negatives, and the strongest model signals. These examples are review aids; the pseudonymous IDs are not exposed as client or URL information.


In [18]:
best_prob = probabilities[best_model_name]
error_frame = model_df.iloc[test_indices][
    ["content_id", "is_declining_label", "impressions_90d", "days_since_last_update", "avg_position", "ctr", "content_age_days"]
].copy()
error_frame["predicted_probability"] = best_prob
error_frame["predicted_label"] = (best_prob >= 0.5).astype(int)
error_frame["error_type"] = np.select(
    [
        (error_frame["predicted_label"] == 1) & (error_frame["is_declining_label"] == 0),
        (error_frame["predicted_label"] == 0) & (error_frame["is_declining_label"] == 1),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)

false_positives = (
    error_frame[error_frame["error_type"] == "false_positive"]
    .sort_values("predicted_probability", ascending=False)
    .head(3)
)
false_negatives = (
    error_frame[error_frame["error_type"] == "false_negative"]
    .sort_values("predicted_probability", ascending=True)
    .head(3)
)

print("Three false positives:")
display(false_positives.round(3))
print("Three false negatives:")
display(false_negatives.round(3))


Three false positives:


,content_id,is_declining_label,impressions_90d,days_since_last_update,avg_position,ctr,content_age_days,predicted_probability,predicted_label,error_type
3982,content_9b4ddfa91f64,0,121,20,8.5,0.00,133,0.790,1,false_positive
29266,content_423c7cf07765,0,1356,20,10.5,0.15,104,0.753,1,false_positive
27764,content_9284688e3982,0,674,20,7.4,0.15,104,0.743,1,false_positive


Three false negatives:


,content_id,is_declining_label,impressions_90d,days_since_last_update,avg_position,ctr,content_age_days,predicted_probability,predicted_label,error_type
17188,content_a0a76e94ade5,1,19,20,9.2,0.0,175,0.0,0,false_negative
25755,content_a8cee66e4788,1,1,20,2.0,100.0,489,0.0,0,false_negative
1068,content_836b4163cf30,1,95,8,28.2,0.0,140,0.0,0,false_negative


In [19]:
# Inspect the strongest signals from the selected model.
best_pipeline = trained_models[best_model_name]
pre = best_pipeline.named_steps["preprocessor"]
estimator = best_pipeline.named_steps["model"]
feature_names = pre.get_feature_names_out()

if hasattr(estimator, "coef_"):
    raw_importance = estimator.coef_[0]
    importance = pd.DataFrame({
        "feature": feature_names,
        "coefficient": raw_importance,
        "absolute_strength": np.abs(raw_importance),
    }).sort_values("absolute_strength", ascending=False)
else:
    raw_importance = estimator.feature_importances_
    importance = pd.DataFrame({
        "feature": feature_names,
        "importance": raw_importance,
    }).sort_values("importance", ascending=False)

print(f"Top signals for {best_model_name}:")
display(importance.head(10).round(4))


Top signals for Logistic Regression:


,feature,coefficient,absolute_strength
3,numeric__word_count,1.0280,1.0280
63,categorical__position_tier_top_3,-0.9115,0.9115
4,numeric__char_count,-0.8689,0.8689
9,numeric__users_90d,-0.7581,0.7581
8,numeric__sessions_90d,0.6838,0.6838
13,numeric__days_with_impressions,0.5897,0.5897
14,numeric__days_with_sessions,-0.4625,0.4625
50,categorical__word_count_tier_<1000,0.4554,0.4554
49,categorical__word_count_tier_3500+,-0.4083,0.4083
37,categorical__main_intent_navigational,-0.4079,0.4079


### Interpretation

The error cases show where the model is uncertain: a page can look risky from its observed visibility, freshness, engagement, or content characteristics without actually receiving the observed decline label, while a declining page can look healthy on those same aggregate signals.

The strongest model features should therefore be treated as **directional associations**, not causal drivers. In particular, a feature ranking is not evidence that changing that feature will cause traffic to recover.

The practical result is a decision-support model: use the ranked probabilities to prioritize human review, then check the actual page and editorial context before taking action.


## 5. Self-check

- [x] Method choice is explained before training.
- [x] Client-held-out validation is used and the random seed is fixed.
- [x] Week-4 baseline and learned models use the same held-out test population.
- [x] Baseline and models are compared with Precision@10/@20/@50.
- [x] ROC-AUC, PR-AUC, precision, recall, and F1 are reported for learned models.
- [x] Label-derived fields and IDs are excluded from the feature matrix.
- [x] Missing numeric values use median imputation with missingness indicators.
- [x] Three concrete false positives and false negatives are inspected.
- [x] Strong model signals are displayed and interpreted carefully.
- [x] Complexity is not rewarded unless it improves the decision-relevant metric.
- [x] No client names, URLs, or private queries are included.
- [ ] Run the notebook top-to-bottom in the repo and commit the executed `work/notebooks/w05_model.ipynb`.
